# HV-Doc-Data — Improved U-Net Document Segmentation (v2)

Run **top to bottom** on Google Colab (`Runtime → Change runtime type → T4 GPU`).

This is an upgraded version of the baseline training notebook. The baseline
trains a small from-scratch U-Net on **one** annotator's labels for **2
epochs** at 256px with **no augmentation**. That leaves a lot on the table
given the eval metric (`Pixel Dice` + **instance-level F1 at IoU ≥ 0.75**) is
strict about boundary precision. This version changes:

| | Baseline | This notebook |
|---|---|---|
| Labels | 1 of 3 annotator CSVs, as-is | **Majority-vote consensus** across all 3 rounds (denoises independent annotator error) |
| Model | Scratch 4-level U-Net, 1.9M params | **U-Net w/ ImageNet-pretrained ResNet34 encoder** (transfer learning), scratch U-Net kept as fallback/baseline |
| Resolution | 256×256 | 384×384 (sharper boundaries → higher IoU) |
| Augmentation | None (train_tf == val_tf) | Flips, small rotation/shift/scale, color jitter, blur, JPEG/noise, coarse dropout |
| Loss | BCE + Dice | BCE (boundary-upweighted) + Dice |
| Schedule | 2 epochs, fixed LR | Warmup + cosine, AdamW, AMP mixed precision, early stopping on best val Dice |
| Threshold | Fixed 0.5 | Swept on validation set to maximize Dice |
| Inference | Single forward pass | **Test-time augmentation** (original + h-flip averaged) |
| Post-processing | Raw contour → polygon | Morphological smoothing + relative-area speckle rejection, keeps top-K instances |

**Before you start:** go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token** — you'll be asked to paste your username and key in the auth cell below.


## 1 — Install dependencies

In [1]:
!pip install -q kagglehub albumentations tqdm segmentation-models-pytorch timm
print("done installing")

done installing


'pip' is not recognized as an internal or external command,
operable program or batch file.


## 2 — Kaggle authentication

Run this cell and follow the prompt to paste your Kaggle username and API key.

In [2]:
import kagglehub

kagglehub.login()

## 3 — Download and copy dataset to `/content/`

`kagglehub` caches to `/root/.cache/...` — not visible in the Colab file browser. This cell downloads the dataset, finds the actual data directory (the Kaggle package nests it under `student/student/`), then copies images and labels into `/content/dataset/` where you can browse them.

In [3]:
import kagglehub, os, shutil, glob

# Step 1 — download to the kagglehub cache (~/.cache/kagglehub/...)
_cache_path = kagglehub.dataset_download("yashvardhangera/hv-doc-data")
print("Kaggle cache path:", _cache_path)

# Step 2 — find the subdirectory that actually contains images/ and labels/
# (the Kaggle package nests the data under student/student/)
def _find_data_root(base):
    for root, dirs, _ in os.walk(base):
        if "images" in dirs and "labels" in dirs:
            return root
    raise RuntimeError(f"Could not find a folder with both images/ and labels/ under {base}")

_data_root = _find_data_root(_cache_path)
print("Data root found at:", _data_root)

# Step 3 — copy into /content/dataset/ so it's visible in the file browser
CONTENT_DATA = "/content/dataset"

if os.path.exists(CONTENT_DATA):
    print(f"{CONTENT_DATA} already exists, skipping copy")
else:
    print("Copying to /content/dataset/ ...")
    shutil.copytree(_data_root, CONTENT_DATA)
    print("Done.")

# Verify
print("\n/content/dataset/ layout:")
for root, dirs, files in os.walk(CONTENT_DATA):
    depth = root[len(CONTENT_DATA):].count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in sorted(files)[:4]:
        print(f"{indent}  {f}")
    if len(files) > 4:
        print(f"{indent}  ... ({len(files)} files total)")

100%|██████████| 886M/886M [02:44<00:00, 5.65MB/s] 

Extracting files...


Kaggle cache path: C:\Users\Surya_prakash\.cache\kagglehub\datasets\yashvardhangera\hv-doc-data\versions\5
Data root found at: C:\Users\Surya_prakash\.cache\kagglehub\datasets\yashvardhangera\hv-doc-data\versions\5\student\student
Copying to /content/dataset/ ...
Done.

/content/dataset/ layout:
dataset/
  images/
    test/
      test_00000.jpg
      test_00001.jpg
      test_00002.jpg
      test_00003.jpg
      ... (1000 files total)
    train/
      train_00000.jpg
      train_00001.jpg
      train_00002.jpg
      train_00003.jpg
      ... (5000 files total)
  labels/
    train_round_1.csv
    train_round_2.csv
    train_round_3.csv


## 4 — Index all images and CSVs

In [4]:
ALL_CSVS = sorted(glob.glob(os.path.join(CONTENT_DATA, "**", "*.csv"), recursive=True))
ALL_IMAGES = sorted(
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.jpg"),  recursive=True) +
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.jpeg"), recursive=True) +
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.png"),  recursive=True)
)

print(f"CSVs  : {len(ALL_CSVS)}")
for p in ALL_CSVS:
    print(" ", p)
print(f"Images: {len(ALL_IMAGES)}  (first few below)")
for p in ALL_IMAGES[:4]:
    print(" ", p)

assert ALL_CSVS,   "No CSV files found under /content/dataset/"
assert ALL_IMAGES, "No image files found under /content/dataset/"

# filename -> full /content/ path
image_lookup = {os.path.basename(p): p for p in ALL_IMAGES}

# test images (used later for pred.csv)
TEST_IMAGE_PATHS = sorted(p for p in ALL_IMAGES if os.path.basename(p).startswith("test_"))
print(f"\nTrain images: {sum(1 for p in ALL_IMAGES if os.path.basename(p).startswith('train_'))}")
print(f"Test  images: {len(TEST_IMAGE_PATHS)}")

CSVs  : 3
  /content/dataset\labels\train_round_1.csv
  /content/dataset\labels\train_round_2.csv
  /content/dataset\labels\train_round_3.csv
Images: 6000  (first few below)
  /content/dataset\images\test\test_00000.jpg
  /content/dataset\images\test\test_00001.jpg
  /content/dataset\images\test\test_00002.jpg
  /content/dataset\images\test\test_00003.jpg

Train images: 5000
Test  images: 1000


## 5 — Load and reconcile ALL THREE annotation rounds

Instead of picking one annotator's CSV (`TRAIN_ROUND = 1/2/3` in the
baseline), we load **all three** and build a **per-pixel majority-vote
consensus mask** for every training image: a pixel counts as "document" if
**≥2 of the 3 annotators** marked it so. This is a simple, principled way to
reduce independent human-labeling noise without arbitrarily trusting one
annotator over another.

We also compute each image's **cross-annotator agreement** (mean pairwise
mask IoU across the 3 rounds) purely as a diagnostic — low-agreement images
are the ones where the annotators disagreed most (often ambiguous or
multi-document images), and are worth spot-checking manually if you have
time.

In [5]:
import pandas as pd, json as _json

round_csvs = {}
for r in (1, 2, 3):
    target_csv = f"train_round_{r}.csv"
    matches = [p for p in ALL_CSVS if os.path.basename(p) == target_csv]
    assert matches, f"Could not find {target_csv} among: {ALL_CSVS}"
    df = pd.read_csv(matches[0]).drop_duplicates(subset="image").reset_index(drop=True)
    round_csvs[r] = df.set_index("image")["polygon"]
    print(f"Round {r}: {len(df)} labeled images  |  {matches[0]}")

# Union of all images labeled in any round
all_labeled_images = sorted(set().union(*[set(s.index) for s in round_csvs.values()]))
print(f"\nTotal unique labeled images across all 3 rounds: {len(all_labeled_images)}")

Round 1: 5000 labeled images  |  /content/dataset\labels\train_round_1.csv
Round 2: 5000 labeled images  |  /content/dataset\labels\train_round_2.csv
Round 3: 5000 labeled images  |  /content/dataset\labels\train_round_3.csv

Total unique labeled images across all 3 rounds: 5000


In [6]:
def _parse_polys(cell):
    if not isinstance(cell, str):
        return []
    try:
        return _json.loads(cell)
    except (TypeError, ValueError, _json.JSONDecodeError):
        return []

records, skipped = [], 0
for name in all_labeled_images:
    if name not in image_lookup:
        skipped += 1
        continue
    round_polygons = [_parse_polys(round_csvs[r].get(name, "[]")) for r in (1, 2, 3)]
    records.append({
        "image_name": name,
        "image_path": image_lookup[name],
        "round_polygons": round_polygons,
    })

if skipped:
    print(f"Skipped {skipped} label rows with no matching image on disk")
print(f"Built {len(records)} consensus-ready training records")

Built 5000 consensus-ready training records


## 6 — Write `seg_common.py`

This is the shared library: models, consensus-mask building, boundary-weighted loss, TTA inference, and post-processing. See the docstring at the top of the file for a summary of what changed vs. the original.

In [7]:
%%writefile /content/seg_common.py
"""Shared code for the document segmentation task (improved version).

Key upgrades vs. the original seg_common.py:
  1. Multi-annotator label reconciliation: builds a per-pixel MAJORITY-VOTE
     consensus mask from the three train_round_*.csv annotation passes,
     instead of training on a single (noisier) annotator's polygons.
  2. A stronger, pretrained-encoder U-Net (segmentation_models_pytorch,
     ResNet34 ImageNet encoder) as the primary model, with the original
     from-scratch 4-level U-Net kept available as `UNetBaseline` for
     comparison / fallback if `segmentation_models_pytorch` isn't installed.
  3. A boundary-weighted BCE + Dice loss, since the eval metric uses a
     strict instance IoU threshold (0.75) -> boundary precision matters a
     lot more than bulk pixel accuracy.
  4. Test-time augmentation (horizontal flip) at inference.
  5. Sturdier mask -> polygon post-processing: morphological smoothing,
     keeps only components above a relative-area threshold (drops speckle
     noise), returns at most a handful of the largest instances.
"""
import glob, json, os, random
import cv2, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset

DATA_ROOT = "/content/dataset"
PRED_ROOT = "/content/results"
CKPT_ROOT = "/content/checkpoints"

IMG_SIZE = 384                 # was 256 -> sharper boundaries help pass a strict IoU threshold
IOU_THR  = 0.75
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


# --------------------------------------------------------------------------- #
# Models
# --------------------------------------------------------------------------- #
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class UNetBaseline(nn.Module):
    """The original ~1.9M-param 4-level U-Net supplied with the assignment.
    Kept as-is so you can still train / compare against the baseline."""
    def __init__(self, base=32):
        super().__init__()
        self.d1 = DoubleConv(3, base);        self.d2 = DoubleConv(base, base*2)
        self.d3 = DoubleConv(base*2, base*4); self.d4 = DoubleConv(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2); self.u3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2); self.u2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base,   2, stride=2); self.u1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.d1(x); e2 = self.d2(self.pool(e1))
        e3 = self.d3(self.pool(e2)); bn = self.d4(self.pool(e3))
        d3 = self.u3(torch.cat([self.up3(bn), e3], 1))
        d2 = self.u2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.u1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


def get_model(name="smp_unet_resnet34", pretrained=True):
    """Factory. Falls back to a deeper scratch U-Net if smp isn't available."""
    if name == "smp_unet_resnet34":
        try:
            import segmentation_models_pytorch as smp
            return smp.Unet(
                encoder_name="resnet34",
                encoder_weights="imagenet" if pretrained else None,
                in_channels=3,
                classes=1,
            )
        except ImportError:
            print("segmentation_models_pytorch not installed -> falling back to UNetBaseline(base=64)")
            return UNetBaseline(base=64)
    if name == "unet_baseline":
        return UNetBaseline(base=32)
    raise ValueError(f"Unknown model name: {name}")


# --------------------------------------------------------------------------- #
# Polygon / mask utilities
# --------------------------------------------------------------------------- #
def polys_to_mask(polygons, H, W):
    mask = np.zeros((H, W), np.uint8)
    for poly in polygons:
        if poly and len(poly) >= 3:
            pts = (np.array(poly, np.float32) * np.array([W, H])).astype(np.int32)
            cv2.fillPoly(mask, [pts], 255)
    return mask


def consensus_mask(mask_list, min_votes=2):
    """Per-pixel majority vote across N annotator masks (same H, W, uint8 0/255).
    min_votes=2 means: a pixel is 'document' if >=2 of the 3 annotators marked it so.
    This is a simple, robust way to denoise independent human annotation error
    without needing to decide any single annotator is 'correct'."""
    stack = np.stack([(m > 127).astype(np.uint8) for m in mask_list], axis=0)
    votes = stack.sum(axis=0)
    return (votes >= min_votes).astype(np.uint8) * 255


def annotator_agreement_iou(mask_list):
    """Mean pairwise IoU across annotators for one image - useful diagnostic
    to spot images where the three annotations wildly disagree."""
    ious = []
    for i in range(len(mask_list)):
        for j in range(i + 1, len(mask_list)):
            a = mask_list[i] > 127; b = mask_list[j] > 127
            inter = np.logical_and(a, b).sum()
            union = np.logical_or(a, b).sum()
            ious.append(inter / union if union > 0 else 1.0)
    return float(np.mean(ious)) if ious else 1.0


def boundary_weight_map(mask, band_px=7, boundary_weight=4.0):
    """Returns a per-pixel loss weight that up-weights a thin band around the
    mask boundary. Since the eval metric needs IoU >= 0.75 per instance,
    getting the boundary right matters far more than bulk interior pixels."""
    mask_u8 = (mask > 0).astype(np.uint8) * 255
    dil = cv2.dilate(mask_u8, np.ones((band_px, band_px), np.uint8))
    ero = cv2.erode(mask_u8, np.ones((band_px, band_px), np.uint8))
    band = (dil > 0) & (ero == 0)
    w = np.ones(mask.shape, np.float32)
    w[band] = boundary_weight
    return w


class CsvSegDatasetConsensus(Dataset):
    """Records: list of {image_name, image_path, round_masks: [polys_r1, polys_r2, polys_r3]}.
    Builds the majority-vote consensus mask + boundary weight map on the fly."""
    def __init__(self, records, transform, min_votes=2):
        self.records = records
        self.transform = transform
        self.min_votes = min_votes

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        img = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        round_masks = [polys_to_mask(polys, H, W) for polys in r["round_polygons"]]
        mask = consensus_mask(round_masks, min_votes=self.min_votes)

        t = self.transform(image=img, mask=mask)
        mask_np = t["mask"].numpy() if torch.is_tensor(t["mask"]) else np.asarray(t["mask"])
        # Recompute the boundary-weight map AFTER resize so the weighted band
        # lines up exactly with the mask resolution the model trains on.
        weight_np = boundary_weight_map(mask_np)

        mask_t = torch.from_numpy((mask_np > 127).astype(np.float32)).unsqueeze(0)
        weight_t = torch.from_numpy(weight_np.astype(np.float32)).unsqueeze(0)
        return t["image"], mask_t, weight_t


# Backwards-compatible simple dataset (single annotation round, no consensus /
# no boundary weighting) - handy for a quick baseline run / comparison.
class CsvSegDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records; self.transform = transform
    def __len__(self): return len(self.records)
    def _raw(self, i):
        r = self.records[i]
        img  = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        mask = polys_to_mask(r["polygons"], H, W)
        return img, mask
    def __getitem__(self, i):
        img, mask = self._raw(i)
        t = self.transform(image=img, mask=mask)
        return t["image"], (t["mask"] > 127).float().unsqueeze(0)


# --------------------------------------------------------------------------- #
# Losses / metrics
# --------------------------------------------------------------------------- #
def dice_coeff(pred, gt, eps=1e-6):
    pred = np.asarray(pred).astype(bool); gt = np.asarray(gt).astype(bool)
    return float((2 * np.logical_and(pred, gt).sum() + eps) / (pred.sum() + gt.sum() + eps))


def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    num  = 2 * (prob * target).sum(dim=(1, 2, 3)) + eps
    den  = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps
    return (1 - num / den).mean()


def weighted_bce_dice_loss(logits, target, weight=None):
    """BCE (optionally up-weighted near the mask boundary) + Dice."""
    if weight is None:
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    else:
        per_px = nn.functional.binary_cross_entropy_with_logits(logits, target, reduction="none")
        bce = (per_px * weight).mean()
    return bce + dice_loss(logits, target)


# --------------------------------------------------------------------------- #
# Inference helpers
# --------------------------------------------------------------------------- #
def preprocess(img_rgb, size=None):
    if size is None: size = IMG_SIZE
    r = cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
    n = (r.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(n.transpose(2, 0, 1)).unsqueeze(0).float()


@torch.no_grad()
def predict_tta(model, img_rgb, size=None, device=None):
    """Average the prediction for the image and its horizontal flip.
    Cheap, reliable accuracy bump with no retraining needed."""
    if size is None: size = IMG_SIZE
    if device is None: device = DEVICE
    x0 = preprocess(img_rgb, size).to(device)
    x1 = preprocess(np.ascontiguousarray(img_rgb[:, ::-1, :]), size).to(device)
    p0 = torch.sigmoid(model(x0))[0, 0].cpu().numpy()
    p1 = torch.sigmoid(model(x1))[0, 0].cpu().numpy()[:, ::-1]
    return (p0 + p1) / 2.0


def mask_to_polygons(binary, size=None, min_area=80, epsilon_frac=0.005,
                      min_area_frac=0.01, max_polygons=4, smooth_ksize=5):
    """Contour -> polygon extraction with extra smoothing / speckle rejection.
    min_area_frac: components smaller than this fraction of the frame area are
    dropped (handles small false-positive blobs the raw min_area pixel
    threshold misses at higher IMG_SIZE)."""
    if size is None: size = IMG_SIZE
    binary = binary.astype(np.uint8)

    if smooth_ksize > 1:
        k = np.ones((smooth_ksize, smooth_ksize), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    frame_area = size * size
    area_thr = max(min_area, min_area_frac * frame_area)

    candidates = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < area_thr:
            continue
        approx = cv2.approxPolyDP(c, epsilon_frac * cv2.arcLength(c, True), True)
        if len(approx) < 3:
            continue
        candidates.append((area, approx))

    candidates.sort(key=lambda t: -t[0])
    polygons = []
    for area, approx in candidates[:max_polygons]:
        pts = approx.reshape(-1, 2).astype(np.float32) / size
        pts = np.clip(pts, 0.0, 1.0)
        polygons.append([[round(float(x), 5), round(float(y), 5)] for x, y in pts])
    return polygons


Writing /content/seg_common.py


### 6a — Annotator agreement diagnostic

Now that `seg_common` is available, quickly check cross-annotator agreement (mean pairwise mask IoU) on a sample of images, to sanity-check the consensus approach before training.

In [9]:
import sys
sys.path.insert(0, "/content")
import seg_common as sc
import cv2, numpy as np, random as _random

_sample = _random.Random(0).sample(records, min(300, len(records)))
agreements = []
for r in _sample:
    H, W = cv2.imread(r["image_path"]).shape[:2]
    masks = [sc.polys_to_mask(polys, H, W) for polys in r["round_polygons"]]
    agreements.append(sc.annotator_agreement_iou(masks))

agreements = np.array(agreements)
print(f"Mean pairwise annotator IoU : {agreements.mean():.3f}")
print(f"Median                      : {np.median(agreements):.3f}")
print(f"5th percentile (worst)      : {np.percentile(agreements, 5):.3f}")
print("\nLow numbers mean annotators disagreed a lot on that image (ambiguous crop, "
      "multi-document scene, etc). The majority-vote consensus mask handles this "
      "gracefully without us having to hand-inspect every image.")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Surya_prakash\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 7 — Imports and config

In [ ]:
import os, sys, random, csv, json
import numpy as np, cv2, torch, torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, "/content")
import seg_common as sc

os.makedirs(sc.CKPT_ROOT, exist_ok=True)
os.makedirs(sc.PRED_ROOT, exist_ok=True)

torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", sc.DEVICE, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 8 — Training hyper-parameters

In [ ]:
MODEL_NAME   = "unet_resnet34_consensus"
MODEL_ARCH   = "smp_unet_resnet34"   # or "unet_baseline" to reproduce the original scratch U-Net
BATCH        = 12
EPOCHS       = 40
MAX_LR       = 3e-4
WARMUP_EPOCHS = 2
WEIGHT_DECAY = 1e-4
VAL_FRAC     = 0.10
NUM_WORKERS  = 2
MIN_VOTES    = 2          # consensus: pixel counts as "document" if >= this many annotators agree
EARLY_STOP_PATIENCE = 8   # stop if val Dice hasn't improved in this many epochs

print(f"model: {MODEL_NAME} ({MODEL_ARCH})  |  img_size {sc.IMG_SIZE}  |  batch {BATCH}  |  epochs {EPOCHS}")

## 9 — Transforms

The baseline's `train_tf` was identical to `val_tf` (resize + normalize only)
— i.e. **no augmentation at all** on 5,000 training images. Photos of
documents vary in lighting, angle, and camera noise, so we add:
flips, mild rotation/shift/scale/perspective (documents are rarely
perfectly axis-aligned), color jitter, blur, and JPEG-compression /
ISO-noise simulation, plus coarse dropout to discourage over-reliance on
any single patch of texture.

In [ ]:
train_tf = A.Compose([
    A.LongestMaxSize(max_size=sc.IMG_SIZE),
    A.PadIfNeeded(sc.IMG_SIZE, sc.IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
    A.Perspective(scale=(0.02, 0.06), p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.MotionBlur(blur_limit=5),
    ], p=0.2),
    A.OneOf([
        A.ISONoise(),
        A.ImageCompression(quality_lower=40, quality_upper=90),
    ], p=0.2),
    A.CoarseDropout(max_holes=4, max_height=int(sc.IMG_SIZE*0.08), max_width=int(sc.IMG_SIZE*0.08), p=0.2),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.LongestMaxSize(max_size=sc.IMG_SIZE),
    A.PadIfNeeded(sc.IMG_SIZE, sc.IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])

## 10 — Build train / val split (consensus labels)

In [ ]:
random.Random(0).shuffle(records)
n_val         = int(len(records) * VAL_FRAC)
val_records   = records[:n_val]
train_records = records[n_val:]

train_ds = sc.CsvSegDatasetConsensus(train_records, train_tf, min_votes=MIN_VOTES)
val_ds   = sc.CsvSegDatasetConsensus(val_records,   val_tf,   min_votes=MIN_VOTES)
train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"train: {len(train_ds)} images  |  val: {len(val_ds)} images")

## 11 — Model, optimizer, schedule

`segmentation_models_pytorch`'s U-Net with an ImageNet-pretrained ResNet34
encoder gives the network a huge head start over training convolutional
filters from scratch on 5,000 images — the encoder already knows how to
detect edges, textures, and gradients, which is most of what "find the
document boundary" needs. AdamW + warmup + cosine decay is a standard,
robust recipe; mixed precision (AMP) roughly doubles training throughput on
a T4 with no accuracy cost.

In [ ]:
model = sc.get_model(MODEL_ARCH, pretrained=True).to(sc.DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_ld)
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
total_steps  = EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=(sc.DEVICE == "cuda"))

def loss_fn(logit, target, weight):
    return sc.weighted_bce_dice_loss(logit, target, weight)

print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params  |  device: {sc.DEVICE}")

## 12 — Train (with AMP, best-checkpoint tracking, early stopping)

In [ ]:
@torch.no_grad()
def eval_dice(loader, thresh=0.5):
    model.eval()
    scores = []
    for x, y, _w in loader:
        with torch.cuda.amp.autocast(enabled=(sc.DEVICE == "cuda")):
            prob = torch.sigmoid(model(x.to(sc.DEVICE)))
        for p, g in zip(prob.cpu().numpy(), y.numpy()):
            scores.append(sc.dice_coeff(p > thresh, g))
    return float(np.mean(scores))


BEST_CKPT_PATH = os.path.join(sc.CKPT_ROOT, f"{MODEL_NAME}_best.pt")
hist = {"train_loss": [], "val_dice": [], "lr": []}
best_val_dice, epochs_no_improve = -1.0, 0

for ep in range(1, EPOCHS + 1):
    model.train()
    running, n = 0.0, 0
    pbar = tqdm(train_ld, desc=f"epoch {ep}/{EPOCHS}", leave=False)
    for x, y, w in pbar:
        x, y, w = x.to(sc.DEVICE), y.to(sc.DEVICE), w.to(sc.DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(sc.DEVICE == "cuda")):
            loss = loss_fn(model(x), y, w)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        running += loss.item(); n += 1
        pbar.set_postfix(loss=running / n, lr=sched.get_last_lr()[0])

    val_dice = eval_dice(val_ld)
    hist["train_loss"].append(running / n)
    hist["val_dice"].append(val_dice)
    hist["lr"].append(sched.get_last_lr()[0])
    print(f"epoch {ep:2d}  loss {running/n:.4f}  val_dice {val_dice:.4f}  lr {sched.get_last_lr()[0]:.2e}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_CKPT_PATH)
        print(f"  -> new best (val_dice={val_dice:.4f}), checkpoint saved")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping: no improvement in {EARLY_STOP_PATIENCE} epochs.")
            break

print(f"\nBest val Dice: {best_val_dice:.4f}  |  checkpoint: {BEST_CKPT_PATH}")

## 13 — Training curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(hist["train_loss"], marker="o"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["val_dice"],   marker="o"); ax[1].set_title("val Dice");   ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1)
ax[2].plot(hist["lr"],         marker="o"); ax[2].set_title("learning rate"); ax[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 14 — Load best checkpoint + sweep the decision threshold

Rather than assume 0.5 is optimal, sweep a range of thresholds on the held-out
validation set and pick the one that maximizes mean Dice. Since Dice/IoU
aren't necessarily maximized at 0.5 (especially with the boundary-weighted
loss), this is a free accuracy gain with zero extra training.

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=sc.DEVICE))
model.eval()
print(f"Loaded best checkpoint: {BEST_CKPT_PATH}  (val_dice={best_val_dice:.4f})")

thresholds = np.arange(0.3, 0.71, 0.05)
scores = [eval_dice(val_ld, thresh=t) for t in thresholds]
BEST_THRESH = float(thresholds[int(np.argmax(scores))])

plt.figure(figsize=(6, 4))
plt.plot(thresholds, scores, marker="o")
plt.axvline(BEST_THRESH, color="red", linestyle="--", label=f"best={BEST_THRESH:.2f}")
plt.xlabel("threshold"); plt.ylabel("val Dice"); plt.legend(); plt.title("Threshold sweep")
plt.show()

print(f"Best threshold: {BEST_THRESH:.2f}  (val Dice {max(scores):.4f})")

---
## 15 — Generate `pred.csv`

You can run this section **independently** after a session reset — just re-run
sections 1, 6, 7 first (data download, `seg_common.py`, imports), then set
`CKPT_PATH` and `MODEL_ARCH` below and pick up from here.

Inference uses **test-time augmentation** (original image + horizontal flip,
probabilities averaged) and the tuned threshold from the previous section,
followed by the smoothed post-processing in `seg_common.mask_to_polygons`.

In [ ]:
# ── Set these if resuming a fresh session ────────────────────────────────────
CKPT_PATH  = BEST_CKPT_PATH if "BEST_CKPT_PATH" in dir() else "/content/checkpoints/unet_resnet34_consensus_best.pt"
MODEL_ARCH = MODEL_ARCH if "MODEL_ARCH" in dir() else "smp_unet_resnet34"
THRESH     = BEST_THRESH if "BEST_THRESH" in dir() else 0.5
# ──────────────────────────────────────────────────────────────────────────────

model = sc.get_model(MODEL_ARCH, pretrained=False).to(sc.DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=sc.DEVICE))
model.eval()
print(f"Loaded: {CKPT_PATH}  |  threshold: {THRESH:.2f}  |  device: {sc.DEVICE}")

In [ ]:
if not TEST_IMAGE_PATHS:
    print("No test images found in /content/dataset/. Check the dataset structure.")
else:
    print(f"Running TTA inference on {len(TEST_IMAGE_PATHS)} test images...")
    rows = []
    for img_path in tqdm(TEST_IMAGE_PATHS, desc="predicting"):
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        prob  = sc.predict_tta(model, image, size=sc.IMG_SIZE, device=sc.DEVICE)
        polygons = sc.mask_to_polygons((prob > THRESH).astype(np.uint8), size=sc.IMG_SIZE)
        rows.append((os.path.basename(img_path), json.dumps(polygons)))

    PRED_CSV = "/content/results/pred.csv"
    with open(PRED_CSV, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image", "polygon"])
        w.writerows(rows)
    print(f"Wrote {len(rows)} rows -> {PRED_CSV}")

### 15a — Sanity-check a few predictions visually

Always eyeball a handful of predictions before submitting — a strong Dice
score on aggregate can still hide systematic failure modes (e.g. always
missing rotated documents, or false-positives on plain backgrounds).

In [ ]:
import random as _random
sample_paths = _random.Random(1).sample(TEST_IMAGE_PATHS, min(6, len(TEST_IMAGE_PATHS)))
pred_lookup = dict(rows)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.ravel(), sample_paths):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    polys = json.loads(pred_lookup[os.path.basename(path)])
    ax.imshow(img)
    for poly in polys:
        pts = np.array(poly) * [W, H]
        pts = np.vstack([pts, pts[0]])
        ax.plot(pts[:, 0], pts[:, 1], "-o", color="lime", linewidth=2, markersize=3)
    ax.set_title(os.path.basename(path)); ax.axis("off")
plt.tight_layout(); plt.show()

## 16 — Submit / download `pred.csv`

In [ ]:
import requests

# ── CONFIG: edit these ────────────────────────────────────────────────────────

BASE_URL        = "http://3.108.8.61:8991"
HIRING_CODE     = "CIT_DE_2026"

PREDICTIONS_CSV = PRED_CSV                 # path to YOUR predictions
NAME            = "Your Full Name"
EMAIL           = "you@university.edu"     # your unique ID -- use the same one every time

# ──────────────────────────────────────────────────────────────────────────────


def submit_predictions(pred_path: str, name: str, email: str, hiring_code: str) -> dict:
    url = f"{BASE_URL}/submit"
    with open(pred_path, "rb") as f:
        files = {"predictions": (pred_path, f, "text/csv")}
        data = {"name": name, "email": email, "hiring_code": hiring_code}
        resp = requests.post(url, data=data, files=files, timeout=300)

    if not resp.ok:
        print(f"Request failed: {resp.status_code}")
        try:
            print(resp.json().get("detail", resp.text))
        except ValueError:
            print(resp.text)
        return {}

    return resp.json()


def print_results(result: dict) -> None:
    if not result:
        return

    s = result["this_submission"]
    b = result["your_best"]

    print()
    print(f'Images evaluated   : {s["n_images"]}')
    print(f'Pixel Dice (mean)  : {s["dice_mean"]:.4f}')
    print(f'Instance Precision : {s["precision"]:.4f}  (TP={s["tp"]}  FP={s["fp"]})')
    print(f'Instance Recall    : {s["recall"]:.4f}  (FN={s["fn"]})')
    print(f'Instance F1        : {s["f1"]:.4f}  (IoU thr = {s["iou_thr"]:.2f})')
    print()
    print(f'Best Dice so far   : {b["dice_mean"]:.4f}  (attempt {b["attempt"]})')
    print(f'Attempts so far    : {result["attempt"]}')
    print(f'Attempts remaining : {result["attempts_remaining"]}')


print_results(submit_predictions(
    PREDICTIONS_CSV,
    NAME,
    EMAIL,
    HIRING_CODE,
))

In [ ]:
from google.colab import files
files.download(PRED_CSV)

## 17 — Further ideas if you have more time/compute

Didn't make the cut for this notebook but worth knowing about:

- **Ensembling across annotator rounds.** Instead of (or in addition to) the
  consensus mask, train 3 separate models — one per `train_round_*.csv` — and
  average their predicted probability maps at inference. This captures each
  annotator's idiosyncrasies rather than collapsing them upfront, at the cost
  of 3x training compute.
- **Larger/stronger encoders** (ResNet50, EfficientNet-B3/B4, or a
  SegFormer/DeepLabV3+ head) if you have a bigger GPU budget than a single T4.
- **Corner-point regression as an auxiliary head.** The dataset provides a
  `corners` column that the baseline pipeline ignores. For (mostly
  quadrilateral) documents, a small auxiliary head that directly regresses
  the 4 corner coordinates — trained jointly with the segmentation loss — can
  sharpen corners beyond what `approxPolyDP` recovers from a soft mask.
- **CRF or guided filtering post-processing** using the original image as
  guidance, to snap the mask boundary to actual image edges.
- **K-fold cross-validation** instead of a single 90/10 split, to get a more
  reliable estimate of generalization and to ensemble fold models.
